#Modelagem Dimensional e Construção da Gold

### 1. Configuração do ambiente

Definição do catálogo e dos schemas utilizados na construção
das tabelas analíticas da camada Gold.

In [0]:
# Importa as funções utilizadas nas transformações
from pyspark.sql import functions as F

# Define o catálogo e os schemas do projeto
catalogo = "workspace"

schema_origem = "silver"

schema_destino = "gold"

print("Ambiente Gold configurado!")

Ambiente Gold configurado!


### 2. Construção da dimensão de clientes

Criação da dimensão de clientes a partir dos dados tratados
na camada Silver.

A tabela contém os identificadores dos clientes e suas
informações geográficas, permitindo análises de pedidos
e entregas por cidade e estado.

In [0]:
# Carrega os clientes tratados da Silver
df_clientes = spark.table(
    f"{catalogo}.{schema_origem}.customers"
)

# Seleciona e organiza os atributos da dimensão
df_dim_clientes = (
    df_clientes

    .select(

        F.col("customer_id"),

        F.col("customer_unique_id"),

        F.col("customer_zip_code_prefix").alias(
            "cep_prefixo"
        ),

        F.col("customer_city").alias(
            "cidade"
        ),

        F.col("customer_state").alias(
            "estado"
        )

    )
)

# Exibe uma amostra da dimensão
display(df_dim_clientes.limit(10))

customer_id,customer_unique_id,cep_prefixo,cidade,estado
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


#### 2.1 Persistência da dimensão

Armazenamento da dimensão de clientes em formato Delta Lake
no schema Gold, disponibilizando os dados para consultas analíticas.

In [0]:
# Salva a dimensão de clientes na camada Gold
(
    df_dim_clientes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.dim_clientes"
    )
)

print("Dimensão de clientes criada na Gold!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.dim_clientes"
    ).count()
)

Dimensão de clientes criada na Gold!
Quantidade de registros: 99441


#### 2.2 Validação da dimensão

Verificação da quantidade de registros, unicidade dos
identificadores e cobertura geográfica da dimensão.

O objetivo é garantir que a transformação preservou
a granularidade original dos registros de clientes.

In [0]:
# Carrega a dimensão persistida
df_dim_clientes_validacao = spark.table(
    f"{catalogo}.{schema_destino}.dim_clientes"
)

# Valida a granularidade e a integridade da dimensão
display(
    df_dim_clientes_validacao.agg(

        F.count("*").alias("total_registros"),

        F.countDistinct(
            "customer_id"
        ).alias("clientes_id_distintos"),

        F.countDistinct(
            "customer_unique_id"
        ).alias("consumidores_unicos"),

        F.countDistinct(
            "estado"
        ).alias("estados_distintos")

    )
)

total_registros,clientes_id_distintos,consumidores_unicos,estados_distintos
99441,99441,96096,27


### 3. Construção da dimensão de produtos

Criação da dimensão de produtos a partir dos dados tratados
e enriquecidos na camada Silver.

A dimensão reúne os identificadores dos produtos, suas
categorias e características físicas.

A categoria analítica utiliza a tradução disponibilizada
pela Olist, preservando os produtos sem categoria identificada.

In [0]:
# Carrega os produtos tratados da camada Silver
df_produtos = spark.table(
    f"{catalogo}.{schema_origem}.products"
)

# Seleciona e organiza os atributos da dimensão de produtos
df_dim_produtos = (
    df_produtos

    .select(

        F.col("product_id"),

        F.col("product_category_name").alias(
            "categoria_original"
        ),

        F.col("categoria_analitica"),

        F.col("product_name_lenght").alias(
            "tamanho_nome"
        ),

        F.col("product_description_lenght").alias(
            "tamanho_descricao"
        ),

        F.col("product_photos_qty").alias(
            "quantidade_fotos"
        ),

        F.col("product_weight_g").alias(
            "peso_gramas"
        ),

        F.col("product_length_cm").alias(
            "comprimento_cm"
        ),

        F.col("product_height_cm").alias(
            "altura_cm"
        ),

        F.col("product_width_cm").alias(
            "largura_cm"
        )

    )
)

# Exibe uma amostra da dimensão
display(df_dim_produtos.limit(10))

product_id,categoria_original,categoria_analitica,tamanho_nome,tamanho_descricao,quantidade_fotos,peso_gramas,comprimento_cm,altura_cm,largura_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,40,287,1,225.00,16.00,10.00,14.00
3aa071139cb16b67ca9e5dea641aaa2f,artes,art,44,276,1,1000.00,30.00,18.00,20.00
96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,46,250,1,154.00,18.00,9.00,15.00
cef67bcfe19066a932b7673e239eb23d,bebes,baby,27,261,1,371.00,26.00,4.00,26.00
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,37,402,4,625.00,20.00,17.00,13.00
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,musical_instruments,60,745,1,200.00,38.00,5.00,11.00
732bd381ad09e530fe0a5f457d81becb,cool_stuff,cool_stuff,56,1272,4,18350.00,70.00,24.00,44.00
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,furniture_decor,56,184,2,900.00,40.00,8.00,40.00
37cc742be07708b53a98702e77a21a02,eletrodomesticos,home_appliances,57,163,1,400.00,27.00,13.00,17.00
8c92109888e8cdf9d66dc7e463025574,brinquedos,toys,36,1156,1,600.00,17.00,10.00,12.00


#### 3.1 Persistência da dimensão de produtos

Armazenamento da dimensão de produtos em formato Delta Lake
no schema Gold.

A tabela será utilizada para enriquecer as análises de vendas
com informações sobre categorias e características dos produtos.

In [0]:
# Salva a dimensão de produtos na camada Gold
(
    df_dim_produtos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.dim_produtos"
    )
)

print("Dimensão de produtos criada na Gold!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.dim_produtos"
    ).count()
)

Dimensão de produtos criada na Gold!
Quantidade de registros: 32951


#### 3.2 Validação da dimensão de produtos

Verificação da integridade da dimensão, considerando
a quantidade de produtos, a unicidade dos identificadores
e a completude das categorias analíticas.

O objetivo é garantir que nenhum produto tenha sido
perdido durante a modelagem dimensional.

In [0]:
# Carrega a dimensão de produtos persistida na Gold
df_dim_produtos_validacao = spark.table(
    f"{catalogo}.{schema_destino}.dim_produtos"
)

# Verifica a integridade dos produtos
display(
    df_dim_produtos_validacao.agg(

        F.count("*").alias("total_produtos"),

        F.countDistinct(
            "product_id"
        ).alias("produtos_unicos"),

        F.countDistinct(
            "categoria_analitica"
        ).alias("categorias_distintas"),

        F.sum(
            F.when(
                F.col("categoria_analitica") == "nao_informada",
                1
            ).otherwise(0)
        ).alias("categorias_nao_informadas"),

        F.sum(
            F.when(
                F.col("categoria_analitica").isNull(),
                1
            ).otherwise(0)
        ).alias("categorias_nulas")

    )
)

total_produtos,produtos_unicos,categorias_distintas,categorias_nao_informadas,categorias_nulas
32951,32951,74,610,0


### 4. Construção da dimensão de vendedores

Criação da dimensão de vendedores a partir dos dados tratados
na camada Silver.

A dimensão contém os identificadores dos vendedores e suas
informações geográficas, permitindo análises de vendas
e entregas por cidade e estado de origem.

In [0]:
# Carrega os vendedores tratados da camada Silver
df_vendedores = spark.table(
    f"{catalogo}.{schema_origem}.sellers"
)

# Seleciona e organiza os atributos da dimensão
df_dim_vendedores = (
    df_vendedores

    .select(

        F.col("seller_id"),

        F.col("seller_zip_code_prefix").alias(
            "cep_prefixo"
        ),

        F.col("seller_city").alias(
            "cidade"
        ),

        F.col("seller_state").alias(
            "estado"
        )

    )
)

# Exibe uma amostra da dimensão
display(df_dim_vendedores.limit(10))

seller_id,cep_prefixo,cidade,estado
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


#### 4.1 Persistência da dimensão de vendedores

Armazenamento da dimensão de vendedores em formato Delta Lake
no schema Gold.

A tabela será utilizada para enriquecer as análises de vendas
com informações geográficas dos vendedores.

In [0]:
# Salva a dimensão de vendedores na camada Gold
(
    df_dim_vendedores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.dim_vendedores"
    )
)

print("Dimensão de vendedores criada na Gold!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.dim_vendedores"
    ).count()
)

Dimensão de vendedores criada na Gold!
Quantidade de registros: 3095


#### 4.2 Validação da dimensão de vendedores

Verificação da quantidade de vendedores, unicidade dos
identificadores e cobertura geográfica.

O objetivo é garantir que a dimensão preserve todos os
vendedores e mantenha a integridade dos dados geográficos.

In [0]:
# Carrega a dimensão persistida na Gold
df_dim_vendedores_validacao = spark.table(
    f"{catalogo}.{schema_destino}.dim_vendedores"
)

# Verifica a integridade da dimensão
display(
    df_dim_vendedores_validacao.agg(

        F.count("*").alias("total_vendedores"),

        F.countDistinct(
            "seller_id"
        ).alias("vendedores_unicos"),

        F.countDistinct(
            "estado"
        ).alias("estados_distintos"),

        F.countDistinct(
            "cidade"
        ).alias("cidades_distintas"),

        F.sum(
            F.when(
                F.col("estado").isNull(),
                1
            ).otherwise(0)
        ).alias("estados_nulos")

    )
)

total_vendedores,vendedores_unicos,estados_distintos,cidades_distintas,estados_nulos
3095,3095,23,611,0


### 5. Construção da dimensão de tempo

Criação de uma dimensão calendário a partir do período de compras
registrado no dataset Olist.

A dimensão contém uma linha por data, incluindo informações
de ano, mês, trimestre e dia da semana.

Essa estrutura permitirá realizar análises temporais de vendas,
pedidos e desempenho das entregas.

In [0]:
# Carrega os pedidos tratados da camada Silver
df_orders = spark.table(
    f"{catalogo}.{schema_origem}.orders"
)

# Identifica a primeira e a última data de compra
df_periodo = df_orders.agg(

    F.min(
        F.to_date("order_purchase_timestamp")
    ).alias("data_inicial"),

    F.max(
        F.to_date("order_purchase_timestamp")
    ).alias("data_final")

)

# Exibe o período identificado
display(df_periodo)

data_inicial,data_final
2016-09-04,2018-10-17


#### 5.1 Geração do calendário

Construção de uma sequência contínua de datas entre a
primeira e a última compra registrada.

Para cada data, são derivados atributos temporais que
facilitam agrupamentos e análises por diferentes períodos.

In [0]:
# Recupera o período de compras
periodo = df_periodo.first()

data_inicial = periodo["data_inicial"]
data_final = periodo["data_final"]

# Gera uma linha para cada data do período
df_calendario = spark.sql(f"""
    SELECT explode(
        sequence(
            DATE('{data_inicial}'),
            DATE('{data_final}'),
            INTERVAL 1 DAY
        )
    ) AS data
""")

# Cria os atributos da dimensão de tempo
df_dim_tempo = (
    df_calendario

    .withColumn(
        "data_id",
        F.date_format("data", "yyyyMMdd").cast("int")
    )

    .withColumn(
        "ano",
        F.year("data")
    )

    .withColumn(
        "mes",
        F.month("data")
    )

    .withColumn(
        "trimestre",
        F.quarter("data")
    )

    .withColumn(
        "dia",
        F.dayofmonth("data")
    )

    .withColumn(
        "dia_semana",
        F.dayofweek("data")
    )

    .withColumn(
        "ano_mes",
        F.date_format("data", "yyyy-MM")
    )

    .select(
        "data_id",
        "data",
        "ano",
        "mes",
        "trimestre",
        "dia",
        "dia_semana",
        "ano_mes"
    )
)

# Exibe uma amostra da dimensão
display(df_dim_tempo.limit(10))

data_id,data,ano,mes,trimestre,dia,dia_semana,ano_mes
20160904,2016-09-04,2016,9,3,4,1,2016-09
20160905,2016-09-05,2016,9,3,5,2,2016-09
20160906,2016-09-06,2016,9,3,6,3,2016-09
20160907,2016-09-07,2016,9,3,7,4,2016-09
20160908,2016-09-08,2016,9,3,8,5,2016-09
20160909,2016-09-09,2016,9,3,9,6,2016-09
20160910,2016-09-10,2016,9,3,10,7,2016-09
20160911,2016-09-11,2016,9,3,11,1,2016-09
20160912,2016-09-12,2016,9,3,12,2,2016-09
20160913,2016-09-13,2016,9,3,13,3,2016-09


#### 5.2 Persistência da dimensão de tempo

Armazenamento da dimensão calendário em formato Delta Lake
no schema Gold.

A tabela será utilizada para apoiar as análises temporais
das tabelas fato.

In [0]:
# Salva a dimensão de tempo na camada Gold
(
    df_dim_tempo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.dim_tempo"
    )
)

print("Dimensão de tempo criada na Gold!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.dim_tempo"
    ).count()
)

Dimensão de tempo criada na Gold!
Quantidade de registros: 774


#### 5.3 Validação da dimensão de tempo

Verificação da quantidade de datas geradas, unicidade
das chaves e cobertura temporal do calendário.

O objetivo é garantir que a dimensão contenha uma linha
por data, sem duplicidades.

In [0]:
# Carrega a dimensão persistida
df_dim_tempo_validacao = spark.table(
    f"{catalogo}.{schema_destino}.dim_tempo"
)

# Verifica a integridade do calendário
display(
    df_dim_tempo_validacao.agg(

        F.count("*").alias("total_datas"),

        F.countDistinct(
            "data_id"
        ).alias("datas_distintas"),

        F.min(
            "data"
        ).alias("primeira_data"),

        F.max(
            "data"
        ).alias("ultima_data"),

        F.countDistinct(
            "ano"
        ).alias("anos_distintos")

    )
)

total_datas,datas_distintas,primeira_data,ultima_data,anos_distintos
774,774,2016-09-04,2018-10-17,3


### 6. Construção da tabela fato de pedidos

Criação da tabela fato_pedidos, com granularidade de uma linha
por pedido.

A tabela consolida informações de compra, entrega e satisfação,
permitindo analisar o desempenho logístico e sua relação com
a experiência dos consumidores.

As avaliações serão agregadas por pedido antes da integração,
evitando multiplicação de registros durante o JOIN.

In [0]:
# Carrega as avaliações tratadas da Silver
df_reviews = spark.table(
    f"{catalogo}.{schema_origem}.order_reviews"
)

# Consolida as avaliações por pedido
df_reviews_agg = (
    df_reviews

    .groupBy("order_id")

    .agg(

        F.round(
            F.avg("review_score"), 2
        ).alias("nota_media_avaliacao"),

        F.count("*").alias(
            "quantidade_avaliacoes"
        ),

        F.sum(
            F.when(
                F.col("possui_comentario") == True,
                1
            ).otherwise(0)
        ).alias("quantidade_comentarios")

    )
)

# Exibe uma amostra das avaliações consolidadas
display(df_reviews_agg.limit(10))

order_id,nota_media_avaliacao,quantidade_avaliacoes,quantidade_comentarios
73fc7af87114b39712e6da79b0a377eb,4.0,1,0
a548910a1c6147796b98fdf73dbeba33,5.0,1,0
f9e4b658b201a9f2ecdecbb34bed034b,5.0,1,0
658677c97b385a9be170737859d3511b,5.0,1,1
8e6bfb81e283fa7e4f11123a3fb894f1,5.0,1,1
b18dcdf73be66366873cd26c5724d1dc,1.0,1,0
e48aa0d2dcec3a2e87348811bcfdf22b,5.0,1,0
c31a859e34e3adac22f376954e19b39d,5.0,1,0
9c214ac970e84273583ab523dfafd09b,5.0,1,0
b9bf720beb4ab3728760088589c62129,4.0,1,1


#### 6.1 Integração dos pedidos e avaliações

Integração dos pedidos tratados na Silver com as avaliações
consolidadas por pedido.

A operação utiliza LEFT JOIN para preservar todos os pedidos,
inclusive aqueles que não possuem avaliação.

São calculados indicadores de prazo de entrega e tempo
decorrido entre a compra e o recebimento.

In [0]:
# Carrega os pedidos tratados da Silver
df_orders = spark.table(
    f"{catalogo}.{schema_origem}.orders"
)

# Integra os pedidos com as avaliações consolidadas
df_fato_pedidos = (
    df_orders.alias("o")

    .join(
        df_reviews_agg.alias("r"),
        "order_id",
        "left"
    )

    # Cria a chave para a dimensão de tempo
    .withColumn(
        "data_id",
        F.date_format(
            "order_purchase_timestamp",
            "yyyyMMdd"
        ).cast("int")
    )

    # Calcula o tempo de entrega em dias
    .withColumn(
        "tempo_entrega_dias",

        F.round(
            (
                F.unix_timestamp("order_delivered_customer_date")
                -
                F.unix_timestamp("order_purchase_timestamp")
            ) / 86400,
            2
        )
    )

    # Calcula o atraso em dias
    .withColumn(
        "atraso_dias",

        F.when(
            F.col("order_delivered_customer_date").isNotNull(),

            F.greatest(
                F.datediff(
                    F.to_date("order_delivered_customer_date"),
                    F.to_date("order_estimated_delivery_date")
                ),
                F.lit(0)
            )
        )
    )

    # Classifica o desempenho da entrega
    .withColumn(
        "status_prazo",

        F.when(
            F.col("order_delivered_customer_date").isNull(),
            "nao_entregue"
        )

        .when(
            F.to_date("order_delivered_customer_date") >
            F.to_date("order_estimated_delivery_date"),
            "atrasado"
        )

        .otherwise("no_prazo")
    )

    # Seleciona os atributos da tabela fato
    .select(

        "order_id",

        "customer_id",

        "data_id",

        "order_status",

        "order_purchase_timestamp",

        "order_delivered_customer_date",

        "order_estimated_delivery_date",

        "tempo_entrega_dias",

        "atraso_dias",

        "status_prazo",

        "nota_media_avaliacao",

        "quantidade_avaliacoes",

        "quantidade_comentarios"

    )
)

# Exibe uma amostra da tabela fato
display(df_fato_pedidos.limit(10))

order_id,customer_id,data_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,tempo_entrega_dias,atraso_dias,status_prazo,nota_media_avaliacao,quantidade_avaliacoes,quantidade_comentarios
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,20171002,delivered,2017-10-02T10:56:33.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,8.44,0,no_prazo,4.0,1,1
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,20180724,delivered,2018-07-24T20:41:37.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,13.78,0,no_prazo,4.0,1,1
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,20180808,delivered,2018-08-08T08:38:49.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,9.39,0,no_prazo,5.0,1,0
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,20171118,delivered,2017-11-18T19:28:06.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,13.21,0,no_prazo,5.0,1,1
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,20180213,delivered,2018-02-13T21:18:39.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,2.87,0,no_prazo,5.0,1,0
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,20170709,delivered,2017-07-09T21:57:05.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,16.54,0,no_prazo,4.0,1,0
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,20170411,invoiced,2017-04-11T12:22:08.000Z,null,2017-05-09T00:00:00.000Z,null,null,nao_entregue,2.0,1,1
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,20170516,delivered,2017-05-16T13:10:30.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,9.99,0,no_prazo,5.0,1,0
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,20170123,delivered,2017-01-23T18:29:09.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,9.82,0,no_prazo,1.0,1,0
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,20170729,delivered,2017-07-29T11:55:02.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,18.22,0,no_prazo,5.0,1,0


#### 6.2 Persistência da tabela fato de pedidos

Armazenamento da tabela fato_pedidos em formato Delta Lake
no schema Gold.

A tabela possui granularidade de uma linha por pedido e
será utilizada nas análises de desempenho logístico
e satisfação dos consumidores.

In [0]:
# Salva a tabela fato de pedidos na Gold
(
    df_fato_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.fato_pedidos"
    )
)

print("Tabela fato_pedidos criada na Gold!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.fato_pedidos"
    ).count()
)

Tabela fato_pedidos criada na Gold!
Quantidade de registros: 99441


#### 6.3 Validação da tabela fato de pedidos

Verificação da granularidade, integridade e classificação
dos prazos de entrega.

A validação confirma se todos os pedidos foram preservados
e identifica a distribuição entre entregas no prazo,
entregas atrasadas e pedidos não entregues.

In [0]:
# Carrega a tabela fato persistida na Gold
df_fato_validacao = spark.table(
    f"{catalogo}.{schema_destino}.fato_pedidos"
)

# Verifica a integridade da tabela fato
display(
    df_fato_validacao.agg(

        F.count("*").alias("total_pedidos"),

        F.countDistinct(
            "order_id"
        ).alias("pedidos_unicos"),

        F.sum(
            F.when(
                F.col("status_prazo") == "atrasado",
                1
            ).otherwise(0)
        ).alias("pedidos_atrasados"),

        F.sum(
            F.when(
                F.col("status_prazo") == "no_prazo",
                1
            ).otherwise(0)
        ).alias("pedidos_no_prazo"),

        F.sum(
            F.when(
                F.col("status_prazo") == "nao_entregue",
                1
            ).otherwise(0)
        ).alias("pedidos_nao_entregues"),

        F.round(
            F.avg("tempo_entrega_dias"), 2
        ).alias("tempo_medio_entrega_dias")

    )
)

total_pedidos,pedidos_unicos,pedidos_atrasados,pedidos_no_prazo,pedidos_nao_entregues,tempo_medio_entrega_dias
99441,99441,6535,89941,2965,12.56


### 7. Construção da tabela fato de itens de pedido

Criação da tabela fato_itens_pedido, com granularidade de
uma linha por item de pedido.

A tabela consolida informações comerciais de cada item,
incluindo produto, vendedor, preço e frete.

Sua finalidade é permitir análises de faturamento,
distribuição geográfica das vendas e desempenho
das categorias de produtos.

In [0]:
# Carrega os itens de pedido tratados na Silver
df_items = spark.table(
    f"{catalogo}.{schema_origem}.order_items"
)

# Carrega os pedidos da Gold para recuperar a dimensão temporal
df_pedidos_gold = spark.table(
    f"{catalogo}.{schema_destino}.fato_pedidos"
)

# Seleciona apenas as informações necessárias dos pedidos
df_pedidos_data = df_pedidos_gold.select(
    "order_id",
    "data_id"
)

# Integra os itens com a dimensão temporal
df_fato_itens = (
    df_items.alias("i")

    .join(
        df_pedidos_data.alias("p"),
        "order_id",
        "left"
    )

    # Calcula o valor total do item, incluindo frete
    .withColumn(
        "valor_total_item",
        F.round(
            F.col("price") + F.col("freight_value"),
            2
        )
    )

    # Seleciona os atributos da tabela fato
    .select(

        "order_id",

        "order_item_id",

        "product_id",

        "seller_id",

        "data_id",

        F.col("price").alias("valor_produto"),

        F.col("freight_value").alias("valor_frete"),

        "valor_total_item"

    )
)

# Exibe uma amostra da tabela fato
display(df_fato_itens.limit(10))

order_id,order_item_id,product_id,seller_id,data_id,valor_produto,valor_frete,valor_total_item
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,20170913,58.90,13.29,72.19
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,20170426,239.90,19.93,259.83
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,20180114,199.00,17.87,216.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,20180808,12.99,12.79,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,20170204,199.90,18.14,218.04
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,20170515,21.90,12.69,34.59
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,20171210,19.90,11.85,31.75
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,20180704,810.00,70.75,880.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,20180319,145.95,11.65,157.60
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,20180702,53.99,11.40,65.39


#### 7.1 Persistência da tabela fato de itens

Armazenamento da tabela fato_itens_pedido em formato
Delta Lake no schema Gold.

A tabela preserva a granularidade original dos itens
e permite análises comerciais sem duplicação de valores.

In [0]:
# Salva a tabela fato de itens na Gold
(
    df_fato_itens.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.fato_itens_pedido"
    )
)

print("Tabela fato_itens_pedido criada na Gold!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.fato_itens_pedido"
    ).count()
)

Tabela fato_itens_pedido criada na Gold!
Quantidade de registros: 112650


#### 7.2 Validação da tabela fato de itens

Verificação da quantidade de itens, unicidade da chave
composta e consistência dos valores financeiros.

A validação também verifica se todos os itens possuem
referência temporal válida.

In [0]:
# Carrega a tabela fato persistida
df_itens_validacao = spark.table(
    f"{catalogo}.{schema_destino}.fato_itens_pedido"
)

# Verifica a integridade da tabela fato
display(
    df_itens_validacao.agg(

        F.count("*").alias("total_itens"),

        F.countDistinct(
            "order_id",
            "order_item_id"
        ).alias("itens_unicos"),

        F.countDistinct(
            "order_id"
        ).alias("pedidos_distintos"),

        F.round(
            F.sum("valor_produto"), 2
        ).alias("valor_total_produtos"),

        F.round(
            F.sum("valor_frete"), 2
        ).alias("valor_total_frete"),

        F.round(
            F.sum("valor_total_item"), 2
        ).alias("valor_total_geral"),

        F.sum(
            F.when(
                F.col("data_id").isNull(),
                1
            ).otherwise(0)
        ).alias("itens_sem_data")

    )
)

total_itens,itens_unicos,pedidos_distintos,valor_total_produtos,valor_total_frete,valor_total_geral,itens_sem_data
112650,112650,98666,13591643.70,2251909.54,15843553.24,0


### 8. Construção da tabela fato de pagamentos

Criação da tabela fato_pagamentos, com granularidade de
uma linha por registro de pagamento.

A tabela consolida os valores financeiros, meios de pagamento
e quantidade de parcelas utilizados pelos consumidores.

Sua finalidade é permitir análises sobre a distribuição
dos meios de pagamento e o comportamento financeiro
dos pedidos.

In [0]:
# Carrega os pagamentos tratados da Silver
df_payments = spark.table(
    f"{catalogo}.{schema_origem}.order_payments"
)

# Recupera a dimensão temporal dos pedidos
df_pedidos_data = spark.table(
    f"{catalogo}.{schema_destino}.fato_pedidos"
).select(
    "order_id",
    "data_id"
)

# Integra os pagamentos com a dimensão temporal
df_fato_pagamentos = (
    df_payments.alias("p")

    .join(
        df_pedidos_data.alias("o"),
        "order_id",
        "left"
    )

    .select(

        "order_id",

        "payment_sequential",

        "data_id",

        "payment_type",

        "payment_installments",

        "payment_value"

    )
)

# Exibe uma amostra
display(df_fato_pagamentos.limit(10))

order_id,payment_sequential,data_id,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,20180425,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,20180626,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,20171212,credit_card,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,20171206,credit_card,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,20180521,credit_card,2,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,1,20180507,credit_card,2,96.12
771ee386b001f06208a7419e4fc1bbd7,1,20170623,credit_card,1,81.16
3d7239c394a212faae122962df514ac7,1,20170605,credit_card,3,51.84
1f78449c87a54faf9e96e88ba1491fa9,1,20180722,credit_card,6,341.09
0573b5e23cbd798006520e1d5b4c6714,1,20170704,boleto,1,51.95


#### 8.1 Persistência da tabela fato de pagamentos

Armazenamento da tabela fato_pagamentos em formato
Delta Lake no schema Gold.

A tabela preserva a granularidade original dos pagamentos,
permitindo analisar múltiplas formas de pagamento
associadas a um mesmo pedido.

In [0]:
# Salva a tabela fato de pagamentos na Gold
(
    df_fato_pagamentos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.fato_pagamentos"
    )
)

print("Tabela fato_pagamentos criada na Gold!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.fato_pagamentos"
    ).count()
)

Tabela fato_pagamentos criada na Gold!
Quantidade de registros: 103886


#### 8.2 Validação da tabela fato de pagamentos

Verificação da quantidade de registros, unicidade
da chave composta e consistência dos valores financeiros.

A validação confirma se os pagamentos foram preservados
durante a transformação da Silver para a Gold.

In [0]:
# Carrega a tabela fato persistida
df_pagamentos_validacao = spark.table(
    f"{catalogo}.{schema_destino}.fato_pagamentos"
)

# Verifica a integridade dos pagamentos
display(
    df_pagamentos_validacao.agg(

        F.count("*").alias("total_pagamentos"),

        F.countDistinct(
            "order_id",
            "payment_sequential"
        ).alias("pagamentos_unicos"),

        F.countDistinct(
            "order_id"
        ).alias("pedidos_distintos"),

        F.round(
            F.sum("payment_value"), 2
        ).alias("valor_total_pagamentos"),

        F.max(
            "payment_installments"
        ).alias("maior_parcelamento"),

        F.sum(
            F.when(
                F.col("data_id").isNull(),
                1
            ).otherwise(0)
        ).alias("pagamentos_sem_data")

    )
)

total_pagamentos,pagamentos_unicos,pedidos_distintos,valor_total_pagamentos,maior_parcelamento,pagamentos_sem_data
103886,103886,99440,16008872.12,24,0
